In [ ]:
# Configuration: change opt-ins here, then run cells in order.
USE_HARDWARE = False  # False: symbolic motors/servos; no GPIO or I2C
RUN_CAMERA = False  # real capture requires USE_HARDWARE and a Pi kernel
ENABLE_POPOUT = False
USE_STREAM_URL = False  # SSH-forward STREAM_PORT; see docs/CAMERA_USAGE.md
STREAM_PORT = 8765
CAMERA_SIZE = (640, 480)
CAMERA_FPS = 15
VIEW_FPS = 10
COMMAND_TIMEOUT = 0.5  # seconds; drive clicks expire, never auto-renew
DRIVE_SPEED = 15  # duty percent; low duty may stall a track
RUN_CONTROLS = False  # True + USE_HARDWARE=False previews a dry-run GUI
ENABLE_SERVOS = False  # also enables tank.arm; no automatic centering
STAND_AND_POWER_CUTOFF_READY = False
SERVO_CLEARANCE_VERIFIED = False
RUN_ARM_EXAMPLE = False
ARM_SAFE_POSE = None  # measured safe ticks: {'arm': ..., 'gripper': ...}
RUN_MANUAL_TEST = False
RUN_COLOR_TEST = False
RUN_OBJECT_TEST = False
TARGET_LABELS = ['bottle']
RUN_FLOOR_CHECK = False
FLOOR_HSV_RANGES = None  # measured [((Hmin,Smin,Vmin),(Hmax,Smax,Vmax))]
RUN_ROAM_TEST = False
MOTION_LIMITS_VERIFIED = False
SUPERVISED_FLAT_ROOM_READY = False

from pathlib import Path
import sys
repo_root = next((p for p in (Path.cwd(), Path.cwd().parent)
                  if (p / 'rasptank' / '__init__.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Open this notebook from the repository or notebooks directory')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from rasptank import CameraController, CameraStream, RaspTank, RoamPolicy
from rasptank.hardware._lifecycle import release
from rasptank.vision import (detect_color, hsv_target, annotate, FloorObserver,
                            MobileNetSSD, detect_objects, find_object,
                            MOBILENET_PROTOTXT_SHA256, MOBILENET_WEIGHTS_SHA256)
print('CONFIG_READY: imports open no hardware')

# RaspTank demo: video, arm and remote controls
Use a **Pi Python kernel**, locally or through Remote SSH/tunneled Jupyter. Defaults open no hardware and download nothing. Stop legacy servers first and read `docs/CAMERA_USAGE.md`. Keep a physical cutoff ready; software cannot stop a dead kernel/OS or hung GPIO. Software results do not validate physical safety.

## 1. Start streaming video (no motors)
Set `USE_HARDWARE = RUN_CAMERA = True` after checking the camera. If pop-ups are blocked, set `USE_STREAM_URL = True` for a loopback video URL instead of widgets. First SSH-forward STREAM_PORT as described in docs/CAMERA_USAGE.md. Open the printed URL manually on your laptop. Closing that page does not stop capture or motors.
Enable only after checking the Pi camera and stopping other owners. This cell returns immediately with live video; later cells remain usable. Set `ENABLE_POPOUT = True` for an optional browser **Pop out video** button (requires anywidget). Nothing opens until clicked. Rerunning closes the old view and pop-out/tank/camera first. `Stop video` stops presentation, not capture or a manually controlled tank.

In [ ]:
release([globals()[name].close for name in ('controls', 'tank', 'view', 'camera')
         if globals().get(name) is not None])
controls = tank = view = camera = arm = None
if RUN_CAMERA:
    if not USE_HARDWARE:
        raise RuntimeError('Real video requires USE_HARDWARE=True and a Pi kernel')
    from rasptank.notebook import show_camera
    camera = CameraController(size=CAMERA_SIZE, fps=CAMERA_FPS, max_age=0.5)
    try:
        camera.start()
        if USE_STREAM_URL:
            view = CameraStream(camera, port=STREAM_PORT, fps=VIEW_FPS)
            print('Open through SSH forwarding:', view.url)
        else:
            view = show_camera(camera, fps=VIEW_FPS, popout=ENABLE_POPOUT)
    except BaseException:
        camera.close()
        raise
else:
    print('CAMERA_DISABLED')

## 2. Robot arm and supervised tank setup
Set `RUN_CONTROLS = True` for the GUI, and `ENABLE_SERVOS = True` for the arm. With `USE_HARDWARE = False`, this is a symbolic preview requiring no Pi drivers. Real setup requires the stand/cutoff check; servos additionally require `SERVO_CLEARANCE_VERIFIED`. No constructor moves/centers servos.

`tank.arm` borrows the same ServoController. The **webServer.py** layout is pan=0, arm=1, hand=2, gripper=3, camera tilt=4. Legacy servo.py labels/signs differ. Values are integer PCA9685 **ticks, not degrees**; initial 300 values are assumptions, not feedback. Verify linkage clearance, channel mapping and a safe absolute pulse before relative steps.

Optional `RUN_ARM_EXAMPLE` applies only your measured `ARM_SAFE_POSE`. Later cells can call `arm.set_pulse('arm', pulse)`, `arm.move_relative('arm', ticks)` (raw signed ticks), or `arm.step('grab', 5)` (named web-control action). Limits raise rather than clip. `arm.center('arm')` explicitly moves to transient calibration, not a known-safe home. Calibration is never persisted. Stop drive/autonomy before manipulating joints.

In [ ]:
if RUN_CONTROLS or RUN_ARM_EXAMPLE or RUN_MANUAL_TEST or RUN_ROAM_TEST:
    if USE_HARDWARE and not STAND_AND_POWER_CUTOFF_READY:
        raise RuntimeError('Complete the stand, wiring and physical-cutoff checks first')
    if USE_HARDWARE and ENABLE_SERVOS and not SERVO_CLEARANCE_VERIFIED:
        raise RuntimeError('Verify servo mapping, safe pulses and linkage clearance first')
    if RUN_ARM_EXAMPLE and (not ENABLE_SERVOS or not ARM_SAFE_POSE):
        raise RuntimeError('Enable servos and supply your measured ARM_SAFE_POSE first')
    release([globals()[name].close for name in ('controls', 'tank', 'view')
             if globals().get(name) is not None])
    controls = tank = view = arm = None
    tank = RaspTank(use_gpio=USE_HARDWARE, enable_servos=ENABLE_SERVOS,
                    enable_ultrasonic=RUN_ROAM_TEST, camera=camera,
                    command_timeout=COMMAND_TIMEOUT)
    try:
        if camera is not None:
            if USE_STREAM_URL:
                view = tank.start_camera_stream(port=STREAM_PORT, fps=VIEW_FPS)
                print('Open through SSH forwarding:', view.url)
            else:
                view = tank.show_camera(fps=VIEW_FPS, popout=ENABLE_POPOUT)
        arm = tank.arm
        if arm is not None:
            print('Assumed/last commanded ticks (not feedback):', arm.positions)
        if RUN_ARM_EXAMPLE:
            # Validate the full requested pose before the first servo output.
            for joint, pulse in ARM_SAFE_POSE.items():
                low, high = arm.limits(joint)
                if type(pulse) is not int or not low <= pulse <= high:
                    raise ValueError(f'{joint}: pulse must be an integer in [{low}, {high}]')
            for joint, pulse in ARM_SAFE_POSE.items():
                arm.set_pulse(joint, pulse)
    except BaseException:
        tank.close()
        raise
else:
    print('TANK_AND_ARM_DISABLED')

## 3. Widget GUI: drive and servo controls
Requires ipywidgets in the kernel. Set `RUN_CONTROLS = True` and run setup above. The GUI starts **disarmed**: Enable driving/Enable servos deliberately. Each drive click expires after `COMMAND_TIMEOUT` seconds; click again for another pulse. There is no held-key worker, continuous drive or automatic renewal after browser disconnection. STOP motors cancels drive/autonomy but leaves servo holding torque unchanged.

All five joints have absolute tick sliders with **Apply** buttons, plus named step buttons and a step-size control. Dragging alone never moves servos. Apply a known-safe absolute pulse before relative steps. Servo actions stop motors/autonomy first. Errors remain visible and disarm the GUI. Rerunning replaces the old GUI and stops motors without closing video/tank. Physical direction, calibration and timing still need validation.

In [ ]:
if RUN_CONTROLS:
    if tank is None:
        raise RuntimeError('Run the supervised tank/arm setup cell first')
    controls = tank.show_controls(speed=DRIVE_SPEED)
else:
    print('WIDGET_CONTROLS_DISABLED')

## 4. Color target and object identification (never drive)
Color targets are not semantic identity. Semantic identification uses the explicit local VOC MobileNet-SSD model described in the usage guide; no weights are fetched by these cells. Select a supported label such as `bottle`, `chair`, `cat`, `dog` or `person`. Recognizing your particular item needs a custom adapter/reference model.

In [ ]:
if RUN_COLOR_TEST:
    if camera is None:
        raise RuntimeError('Enable camera preview first')
    from IPython.display import Image, display
    frame = camera.capture()
    matches = detect_color(frame, hsv_ranges=hsv_target(0), min_area=100, label='red target')
    print(matches)
    display(Image(data=annotate(frame, matches)))
else:
    print('COLOR_TEST_DISABLED')

In [ ]:
if RUN_OBJECT_TEST:
    if camera is None:
        raise RuntimeError('Enable camera preview first')
    from IPython.display import Image, display
    detector = MobileNetSSD(repo_root / 'models/deploy.prototxt',
                            repo_root / 'models/mobilenet_iter_73000.caffemodel',
                            prototxt_sha256=MOBILENET_PROTOTXT_SHA256,
                            weights_sha256=MOBILENET_WEIGHTS_SHA256)
    frame = camera.capture()
    matches = detect_objects(frame, detector=detector, labels=TARGET_LABELS, confidence=0.5)
    print(matches)
    display(Image(data=annotate(frame, matches)))
    # Optional bounded multi-frame search, returning structured results:
    # matches = find_object(camera, detector=detector, labels=TARGET_LABELS, timeout=5)
else:
    print('OBJECT_TEST_DISABLED')

## 5. Manual motion with live video
First secure the tank on a stand with tracks clear, verify wiring/direction and keep the cutoff ready. The configured COMMAND_TIMEOUT lease stops a forgotten command; it is not a hardware emergency stop. Later separate cells can call `tank.forward(15)`, `tank.turn_left(15)`, `tank.stop()` while the view updates. Direct `tank.motors` calls bypass supervision.

In [ ]:
if RUN_MANUAL_TEST:
    if tank is None:
        raise RuntimeError('Run the supervised tank setup first')
    tank.forward(DRIVE_SPEED)  # watchdog expires this command
else:
    print('MANUAL_MOTION_DISABLED')

## 6. Calibrated visual corridor, then finite room trial
Set `FLOOR_HSV_RANGES` from your actual illuminated floor (OpenCV H=0..179, S/V=0..255). Do not use a generic range for navigation. Verify the lower center corridor corresponds to the forward ultrasonic beam, with the camera fixed and aligned. Inspect `observer(camera.capture())` while stationary with clear floor, obstacles and poor light; any unknown must stop the tank.

Only after measuring Pi sensor age, stopping travel and watchdog latency, set the motion limits and enable a **supervised** floor trial. Fix the camera at its verified pose; recalibrate after any servo change that alters camera/sensor alignment. The trial has no servo movement, rear travel or automatic model download. The recovery budget and duration are finite. Close the GUI before roaming. `tank.stop()` cancels roaming; manual facade motion cancels it before taking over.

In [ ]:
observer = None
if RUN_FLOOR_CHECK:
    if camera is None or FLOOR_HSV_RANGES is None:
        raise RuntimeError('Enable camera and supply measured floor ranges first')
    observer = FloorObserver(FLOOR_HSV_RANGES, roi=(0.3, 0.55, 0.7, 0.95))
    print(observer(camera.capture()))
else:
    print('FLOOR_CHECK_DISABLED')

In [ ]:
if RUN_ROAM_TEST:
    if (not USE_HARDWARE or tank is None or tank.ultrasonic is None or camera is None
            or observer is None or not (MOTION_LIMITS_VERIFIED and SUPERVISED_FLAT_ROOM_READY)):
        raise RuntimeError('Complete camera/floor calibration, measured limits and room safety checks first')
    if controls is not None:
        controls.close()
        controls = None
    policy = RoamPolicy(speed=20, stop_distance=0.5, resume_distance=0.6,
                        max_age=0.3, pulse=0.1, max_turns=12, max_duration=10)
    session = tank.start_roaming(observer=observer, policy=policy)
    print(session.status())  # returns immediately; inspect tank.roaming_status() in another cell
else:
    print('ROAMING_DISABLED')

## Stop and cleanup
Run this cell before leaving the notebook or switching owners. It stops controls/tank, releases commanded servo channels (support the arm first), closes presentation, then closes the caller-owned camera. A latched fault remains visible in `tank.roaming_status()`; inspect its cause before explicitly calling `tank.reset_drive_fault()`. A stuck custom observer must exit before restarting autonomy. Kernel/OS failure still requires the physical cutoff.

In [ ]:
release([globals()[name].close for name in ('controls', 'tank', 'view', 'camera')
         if globals().get(name) is not None])
controls = tank = view = camera = arm = None
print('CLEANUP_DONE')